# Speculative Decoding in vLLM: Measure Your Acceptance Rate Before You Trust the Speedup

Companion notebook for the GDE #5 post (Fri Nov 20, 2026). vLLM 0.33.0, PyTorch 2.14.

Speculative decoding is lossless — rejection sampling guarantees the output
distribution is unchanged — but it is **not** free. Whether it pays is decided by
two numbers you can measure on your own hardware: the per-token acceptance rate
`a`, and the draft-to-target cost ratio `c`. This notebook computes the acceptance
math, finds your optimal draft length, finds the breakeven acceptance rate for
three draft costs, reproduces the batch-size collapse, and emits ready-to-paste
`--speculative-config` JSON for vLLM 0.33.0.

Field numbers cited (vendor/paper-reported, flagged in the post): EAGLE-3 paper
(arXiv 2503.01840), youngju.dev 500-prompt benchmark, arXiv 2508.08192 batch study,
arXiv 2406.14066 serving study, DiffVibe 0.33 lab, Spheron H200 numbers.

## 1. The acceptance math — exact, no fitting

One verifier pass checks `N` drafted tokens. If each drafted token is accepted
with probability `a` (independent for the toy model), the expected number of
tokens you keep is the truncated geometric sum:

$$E[\text{accepted}] = \frac{1 - a^{N+1}}{1 - a}$$

The `+1` is the bonus token: when all `N` drafts are accepted, the verifier
still samples one extra token for free.

In [ ]:
def expected_accepted(a, N):
    """Expected tokens kept per verifier pass. Exact formula."""
    return (1 - a ** (N + 1)) / (1 - a)

# DiffVibe 0.33 lab check points: k=4, a=0.8 -> ~3.4 tokens; a=0.5 -> ~1.9
assert abs(expected_accepted(0.8, 4) - 3.37) < 0.01, expected_accepted(0.8, 4)
assert abs(expected_accepted(0.5, 4) - 1.94) < 0.01, expected_accepted(0.5, 4)

print("Expected accepted tokens per verifier pass")
print(f"{'a':>5} | " + " | ".join(f"N={N:<5}" for N in (1, 2, 3, 5, 7)))
for a in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    row = " | ".join(f"{expected_accepted(a, N):5.2f}" for N in (1, 2, 3, 5, 7))
    print(f"{a:5.2f} | {row}")
print()
print("At a=0.5, going N=3 -> N=7 buys 1.88 -> 1.99 tokens: +0.11 for 4 extra")
print("draft tokens. Past the knee, longer drafts are pure waste.")

## 2. Speedup model — and the optimal draft length

Wall time per accepted token is `(N*c + 1) / E[accepted]`, where `c` is the draft
cost as a fraction of one target pass. Baseline (no speculation) is 1.0 time units
per token, so `speedup = E[accepted] / (N*c + 1)`. The optimal `N` grows with `a`
and shrinks with `c` — there is no universal draft length.

In [ ]:
def speedup(a, N, c):
    return expected_accepted(a, N) / (N * c + 1)

def optimal_N(a, c, n_max=12):
    return max(range(1, n_max + 1), key=lambda N: speedup(a, N, c))

# EAGLE head is cheap: c ~ 0.05. A 1B draft model for an 8B target: c ~ 0.10-0.20.
for c in (0.05, 0.10, 0.20):
    print(f"draft cost c = {c}:")
    for a in (0.5, 0.6, 0.7, 0.8, 0.9):
        n_star = optimal_N(a, c)
        print(f"  a={a:.1f}: optimal N={n_star:2d}, speedup {speedup(a, n_star, c):.2f}x"
              f"  (N=5 gives {speedup(a, 5, c):.2f}x)")
    print()

# Sanity: EAGLE-3 at a=0.81, N=5, c=0.06 should land near the independent
# benchmark's measured 2.89x (youngju.dev, 500 prompts) — within 5%.
model = speedup(0.81, 5, 0.06)
assert abs(model - 2.89) / 2.89 < 0.05, model
print(f"model check: EAGLE-3 a=0.81, N=5, c=0.06 -> {model:.2f}x vs measured 2.89x  OK")

## 3. The 50% cliff — find your breakeven acceptance rate

Below some acceptance rate the draft costs more than the hits save, and
speculation is a net loss. The breakeven depends on draft cost: a cheap EAGLE
head survives lower acceptance than a full draft model. The practitioner rule
of thumb (~50%) sits between the two — measure, don't assume.

In [ ]:
import numpy as np

def breakeven_a(c, N=5):
    """Lowest acceptance rate where speedup >= 1.0 (binary search)."""
    lo, hi = 0.01, 0.99
    for _ in range(60):
        mid = (lo + hi) / 2
        if speedup(mid, N, c) >= 1.0:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

for c, label in ((0.05, "EAGLE head"), (0.10, "typical head"),
                 (0.20, "small draft model")):
    b = breakeven_a(c)
    print(f"{label:20s} c={c:.2f}: breakeven a = {b:.3f}  "
          f"(speedup at a=0.50: {speedup(0.50, 5, c):.2f}x,"
          f" at a=0.80: {speedup(0.80, 5, c):.2f}x)")

# Assertions: toy-model breakevens. Note the draft-model case (c=0.20) lands
# at ~0.51 — that is where the field's ~50% rule of thumb comes from. The
# toy model omits real overheads (verification of rejected tokens, draft
# scheduling, KV pressure), so treat 0.50 as the operational floor for all three.
assert 0.15 < breakeven_a(0.05) < 0.25
assert 0.30 < breakeven_a(0.10) < 0.38
assert 0.45 < breakeven_a(0.20) < 0.55
print()
print("With a real draft model, a < ~0.51 means speculation is slower than")
print("doing nothing — and overhead pushes the EAGLE-head breakeven up too.")
print("If your mean acceptance length is under ~2.0 tokens/step (alpha under")
print("~0.50), turn speculation off for that workload.")

## 4. The batch-size collapse

Speculation exploits idle GPU compute while decode is memory-bound. As the batch
grows the GPU becomes compute-bound on its own and verification turns into
overhead. "Efficient Speculative Decoding for Llama at Scale" (arXiv:2508.08192)
measured EAGLE on Llama-3-70B falling from **1.96x at batch 1 to 1.21x at batch
128**. The curve below is an exponential decay fitted exactly through those two
reported endpoints — an interpolation, not measured points. The serving study
(arXiv:2406.14066) adds the request-rate version: past ~12 req/s, 5-token
proposals stop helping; past ~16 req/s, even 3-token proposals slow you down.

In [ ]:
# Exponential decay through the two reported endpoints (arXiv:2508.08192)
k = -np.log(0.21 / 0.96) / 127   # s(b) = 1 + 0.96 * exp(-k*(b-1))
def spec_speedup_batch(b):
    return 1 + 0.96 * np.exp(-k * (b - 1))

assert abs(spec_speedup_batch(1) - 1.96) < 1e-9
assert abs(spec_speedup_batch(128) - 1.21) < 1e-9

for b in (1, 2, 4, 8, 16, 32, 64, 128):
    print(f"batch {b:>4d}: {spec_speedup_batch(b):.2f}x")
print()
print("At batch 32 the fitted curve is already down to ~1.66x; at 64, ~1.45x.")
print("vLLM 0.33 ships disable_by_batch_size for exactly this reason: turn")
print("speculation off above the batch where it stops paying on your hardware.")

## 5. Measure your own acceptance rate — and know which of the three numbers you're reading

Three different numbers all get called "acceptance rate", and mixing them up is
the most common way to misread a speculative-decoding dashboard:

- **α, the per-token acceptance probability** (the papers' number): given all
  previous drafts accepted, the chance the next one is too. EAGLE-3 reports
  α ≈ 0.80–0.88 on chat/code.
- **`draft_acceptance_rate`** (vLLM's number): `num_accepted_draft_tokens /
  num_draft_tokens` — accepted over *all* proposed draft tokens. In the
  sequential model that is E[n]/N, which is always lower than α: at α = 0.82,
  N = 5 it reads ≈ 0.57. Same drafter, longer draft, lower reading.
- **`mean_acceptance_length`**: `1 + num_accepted_draft_tokens /
  num_spec_steps` — tokens emitted per verify step including the bonus token.
  This is E[kept] from section 1, and it is the number the speedup actually
  depends on. vLLM exposes all three: per-request in
  `metrics.speculative_decoding` (`--per-request-spec-decode-metrics`), and
  server-aggregated at `/metrics` as
  `vllm:spec_decode_num_accepted_tokens_total` /
  `vllm:spec_decode_num_draft_tokens_total` (docs.vllm.ai, acceptance-metrics
  page; per-request shape from PR #48915).

The cell below simulates a measurement run and computes all three from the
same per-step accepted counts, so the mapping is explicit.

In [ ]:
rng = np.random.default_rng(7)

def simulate_steps(a_true, N, steps=20000):
    """Per-step accepted-draft counts; a_true = per-token accept prob."""
    accepted = []
    for _ in range(steps):
        n = 0
        while n < N and rng.random() < a_true:
            n += 1
        accepted.append(n)
    return np.array(accepted)

print(f"{'alpha':>6} | {'vLLM rate':>9} | {'mean acc. len':>13} | {'alpha-hat':>9}")
for a_true in (0.55, 0.70, 0.82, 0.90):
    acc = simulate_steps(a_true, N=5)
    steps = len(acc)
    # vLLM's draft_acceptance_rate: accepted / ALL proposed draft tokens
    vllm_rate = acc.sum() / (steps * 5)
    # mean acceptance length: 1 + accepted / steps (bonus token included)
    mean_len = 1 + acc.mean()
    # conditional alpha-hat: accepted / positions actually evaluated
    # (a rejected token was evaluated; tokens past the first rejection were not)
    positions = acc + (acc < 5)
    alpha_hat = acc.sum() / positions.sum()
    print(f"{a_true:6.2f} | {vllm_rate:9.3f} | {mean_len:13.2f} | {alpha_hat:9.3f}")
    # the conditional estimator recovers the true per-token probability...
    assert abs(alpha_hat - a_true) < 0.02, (a_true, alpha_hat)
    # ...and the mean acceptance length matches the section-1 formula exactly
    assert abs(mean_len - expected_accepted(a_true, 5)) < 0.05, (a_true, mean_len)

print()
print("Same drafter, three readings: at alpha=0.82, vLLM's rate says 0.57 while")
print("the mean acceptance length says ~3.9 tokens/step. Judge the drafter by")
print("mean acceptance length; judge the config by the speedup model in §2-3.")

## 6. Emit your `--speculative-config` (vLLM 0.33.0)

One JSON argument configures everything. The generator below validates the
choices vLLM 0.33 actually enforces: Medusa was **removed** in 0.33.0, MTP needs
no separate draft model (the heads ship inside the target checkpoint), and
`eagle3` / `draft_model` need a `model` path. `num_speculative_tokens` starts
at 3–5 and is swept, never guessed.

In [ ]:
import json

SUPPORTED_033 = {"ngram", "suffix", "eagle", "eagle3", "mtp", "dflash",
                 "dspark", "draft_model"}
REMOVED_033 = {"medusa", "mlp"}  # Medusa + MLP speculators removed in 0.33.0

def speculative_config(method, model=None, num_speculative_tokens=5,
                       **extra):
    if method in REMOVED_033:
        raise ValueError(f"{method} was REMOVED in vLLM 0.33.0 — use eagle3/mtp/dspark")
    if method not in SUPPORTED_033:
        raise ValueError(f"unknown method {method!r} for vLLM 0.33.0")
    if not (1 <= num_speculative_tokens <= 8):
        raise ValueError("start at 3-5 and sweep; >8 wastes verification")
    if method in ("eagle", "eagle3", "draft_model", "dflash", "dspark") and not model:
        raise ValueError(f"{method} needs a draft model path")
    if method == "mtp" and model is not None:
        raise ValueError("mtp heads ship inside the target checkpoint — no model arg")
    cfg = {"method": method, "num_speculative_tokens": num_speculative_tokens}
    if model:
        cfg["model"] = model
    cfg.update(extra)
    return cfg

configs = {
    # Zero-effort row: copy-heavy output (code edits, JSON extraction), no model.
    "ngram (free)": speculative_config("ngram", num_speculative_tokens=3,
                                       prompt_lookup_min=3, prompt_lookup_max=5),
    # Small same-tokenizer sibling, e.g. Llama-3.1-1B drafting for Llama-3.1-8B.
    "draft model": speculative_config("draft_model",
                                      model="meta-llama/Llama-3.1-1B-Instruct",
                                      num_speculative_tokens=5),
    # Trained head on the target's hidden states — highest acceptance on chat.
    "eagle3": speculative_config("eagle3",
                                 model="yuhuili/EAGLE3-LLaMA3.3-Instruct-70B",
                                 num_speculative_tokens=5),
    # Heads built into the checkpoint: Qwen3.5, DeepSeek-V3/R1, GLM-5, Gemma 4.
    "mtp (built-in)": speculative_config("mtp", num_speculative_tokens=1),
}
for name, cfg in configs.items():
    print(f"# {name}")
    print("vllm serve <TARGET> --speculative-config "
          f"'{json.dumps(cfg)}'")
    print()

# Guard rails that must hold for every config we emit
try:
    speculative_config("medusa", model="x")
    raise AssertionError("medusa should have been rejected")
except ValueError as e:
    print("medusa correctly rejected:", e)
try:
    speculative_config("mtp", model="x")
    raise AssertionError("mtp with a model arg should have been rejected")
except ValueError as e:
    print("mtp + model correctly rejected:", e)
print()
print("Decision rule: copy-heavy output -> ngram (free). General chat/code at")
print("low concurrency -> eagle3. Model ships MTP heads -> mtp (zero extra")
print("artifact). Saturated serving -> nothing; use disable_by_batch_size.")